# HF-001 — Installing Hugging Face Transformers

Welcome to the first lesson of the **Hugging Face Fundamentals** module. By the end of this notebook you will have a working `transformers` installation and a verified environment you can build on for the rest of the course.

> **What is Transformers?** The `transformers` library by Hugging Face gives you thousands of pre-trained models (text, vision, audio) behind a single Python API — download a model by name and use it in three lines of code.

**What this notebook covers:**

1. What the Transformers ecosystem is (and why each piece matters)
2. Setting up a virtual environment
3. Installing `transformers` and `torch` (pip *and* conda)
4. Verifying the installation (versions, imports, GPU)
5. Running a tiny end-to-end smoke test
6. Troubleshooting common problems

## The Transformers ecosystem — what you are installing

Beginners often think of `transformers` as a single package. In practice it is the center of a small ecosystem. This is what you are really installing:

| Package | What it does | Needed for |
|---|---|---|
| `transformers` | The main library: pipelines, model classes, tokenizers | Every lesson |
| `torch` (PyTorch) | The *engine* that runs the models (tensor math + GPU support) | Every lesson |
| `tokenizers` | Ultra-fast tokenization (installed automatically with `transformers`) | Automatic |
| `huggingface_hub` | Downloads models from the Hub and manages the cache | Automatic |
| `safetensors` | Safe, fast format for storing model weights | Automatic |
| `sentencepiece` | Tokenizer helper for multilingual / BPE models (T5, Llama, …) | Some models |
| `datasets`, `accelerate` | Data loading and GPU training helpers | Later lessons |

**Mental model:** `transformers` is the steering wheel, `torch` is the engine, and the Hub (via `huggingface_hub`) is the fuel station — models are downloaded on demand and cached on your disk.

## Prerequisites

- **Python 3.8+** — check with `python --version` (download from [python.org](https://www.python.org/downloads/))
- **pip** — the Python package manager (bundled with Python)
- **Internet access** — packages and model weights are downloaded on demand
- *(optional but recommended)* a **CUDA-capable NVIDIA GPU** for fast training and inference

No GPU? No problem — every model used in this course has a CPU-friendly variant.

## Step 0 — Create a virtual environment

A virtual environment keeps this course's packages isolated from the rest of your system. Run the two commands below in a terminal:

```bash
python -m venv .venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate
```

> Tip: in VS Code, select the interpreter from `.venv` via *Command Palette → Python: Select Interpreter*.

> 💡 **One-click alternative:** you can skip manual installation entirely — the companion script installs whatever is missing for you:
>
> ```bash
> python 01_Installing_Transformers.py --install --smoke
> ```

## Step 1 — Install the packages

Run the cell below (or the equivalent `pip install` in your terminal). `%pip` is the Jupyter magic that always installs into the kernel's environment — use it instead of plain `!pip` inside notebooks:

In [ ]:
%pip install transformers torch

**Alternative — conda** (if you use Anaconda/Miniconda, the GPU-enabled `pytorch` is easier to install):

```bash
conda create -n llm-course python=3.10 -y
conda activate llm-course
conda install pytorch torchvision torchaudio pytorch-cuda -c pytorch -c nvidia
pip install transformers
```

**For the full course dependency set** (Modules 1–2), install `requirements.txt` instead:

```python
%pip install -r requirements.txt
```

## Step 2 — Verify the environment

The cell below mirrors the companion script `01_Installing_Transformers.py`. Run it to confirm Python, `transformers`, and `torch` are all present:

In [ ]:
import platform
import sys

print(f'Python      : {platform.python_version()}')
print(f'Requirement : Python 3.8+ ->', 'OK' if sys.version_info >= (3, 8) else 'TOO OLD')

try:
    import transformers
    print(f'transformers: {transformers.__version__}  OK')
except ImportError:
    print('transformers: NOT installed — run %pip install transformers')

try:
    import torch
    cuda = torch.cuda.is_available()
    device = torch.cuda.get_device_name(0) if cuda else 'CPU'
    print(f'torch       : {torch.__version__}  OK')
    print(f'cuda        : available={cuda}  device={device}')
except ImportError:
    print('torch       : NOT installed — run %pip install torch')

## Step 3 — Check for a GPU

PyTorch automatically uses the GPU when one is available. If `available=True` below, the rest of the course will be noticeably faster:

In [ ]:
import torch

print('CUDA available :', torch.cuda.is_available())
print('GPU device     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU mode)')

## Step 4 — End-to-end smoke test

The real proof that the installation works: download a small sentiment model and run one prediction. This uses the canonical `distilbert` SST-2 model (~250 MB, one-time download, cached afterwards) and needs no GPU:

In [ ]:
from transformers import pipeline

classifier = pipeline(
    'sentiment-analysis',
    model='distilbert/distilbert-base-uncased-finetuned-sst-2-english',
)

result = classifier('The Hugging Face ecosystem is awesome!')[0]
print(f'{result["label"]} with confidence {result["score"]:.3f}')

## Troubleshooting

| Symptom | Fix |
|---|---|
| `pip: command not found` | Use `python -m pip` instead, or reinstall Python with *Add to PATH* |
| `ModuleNotFoundError: No module named 'transformers'` | Run `%pip install transformers` in the **same kernel**, or activate the right venv |
| Packages install but the notebook still can't import them | The notebook kernel and the terminal use different environments — run `%pip install` inside the notebook |
| Install fails on Windows (build tools) | Install `torch` first from [pytorch.org](https://pytorch.org/get-started/locally/), then `transformers` |
| Slow on CPU | Expected for big models — prefer the small `distilbert/*` models used in this course |
| CUDA out of memory | Use a smaller model, reduce batch size, or fall back to CPU |
| `sentencepiece` errors | `%pip install sentencepiece` (needed by T5/Llama-style tokenizers) |
| Warning about unauthenticated HF requests | Optional: set a free `HF_TOKEN` from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) for higher rate limits |

## Resources & next steps

- Official docs: [huggingface.co/docs/transformers](https://huggingface.co/docs/transformers)
- Installation guide: [huggingface.co/docs/transformers/installation](https://huggingface.co/docs/transformers/installation)
- PyTorch install (pick your OS/CUDA): [pytorch.org/get-started/locally](https://pytorch.org/get-started/locally/)
- Hugging Face Course: [huggingface.co/learn/nlp-course](https://huggingface.co/learn/nlp-course)
- A curated list of links lives in `resources/reference_links.md`

**Quick checks, outside the notebook:**

```bash
python 01_Installing_Transformers.py            # check the environment
python 01_Installing_Transformers.py --install  # install anything missing
python 01_Installing_Transformers.py --smoke    # run the end-to-end test
```

**Next lesson:** HF-002 — Hugging Face Pipelines (`02_HuggingFace_Pipeline`), where we run our first real predictions.

---

*HF-001 · Hugging Face Fundamentals · Generative AI & LLM Mastery Course*